# PyTorch Baseline CNN Model

Training a baseline Convolutional Neural Network using PyTorch with GPU acceleration.

**Key Improvements:**
- Optimized learning rate (0.0001) for stable convergence
- Data augmentation (horizontal flip, brightness adjustment)
- Longer training with proper early stopping

**Objectives:**
- Build and train baseline CNN with PyTorch on GPU
- Achieve robust performance on facial emotion recognition
- Save model for inference
- Demonstrate proper training practices


## 1. Setup and Imports

In [2]:
import os
import sys
import json
import importlib
import numpy as np
import torch
import torch.nn as nn
torch.cuda.empty_cache()
# Add src to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

# Clear any cached imports
for module in list(sys.modules.keys()):
    if module.startswith('src'):
        del sys.modules[module]

# Import with fresh modules
from src.pytorch_models import get_model
from src.pytorch_train import train_model, create_dataloaders, plot_training_history
from src.pytorch_evaluate import create_evaluation_report

# Set seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"Device: {device}")
print(f"\n✓ All imports successful")

PyTorch version: 2.5.1
GPU Available: True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 4.29 GB
Device: cuda

✓ All imports successful


## 2. Load Preprocessed Data

In [3]:
# Load preprocessed data (same as Keras version)
data_dir = os.path.join('..', 'data', 'preprocessed')

X_train = np.load(os.path.join(data_dir, 'X_train.npy'))
X_val = np.load(os.path.join(data_dir, 'X_val.npy'))
X_test = np.load(os.path.join(data_dir, 'X_test.npy'))
y_train = np.load(os.path.join(data_dir, 'y_train.npy'))
y_val = np.load(os.path.join(data_dir, 'y_val.npy'))
y_test = np.load(os.path.join(data_dir, 'y_test.npy'))

# Load class weights
with open(os.path.join(data_dir, 'class_weights.json'), 'r') as f:
    class_weights_dict = json.load(f)

# Convert to PyTorch tensor
class_weights_list = [class_weights_dict[str(i)] for i in range(7)]
class_weights = torch.tensor(class_weights_list, dtype=torch.float32, device=device)

print(f"Training data: {X_train.shape}")
print(f"Validation data: {X_val.shape}")
print(f"Test data: {X_test.shape}")
print(f"Y_train shape: {y_train.shape}")
print(f"\nClass weights: {class_weights_list}")

Training data: (22967, 48, 48, 1)
Validation data: (5742, 48, 48, 1)
Test data: (7178, 48, 48, 1)
Y_train shape: (22967,)

Class weights: [1.0265957446808511, 9.401146131805158, 1.0012206286237413, 0.5684338184338185, 0.8260322255790534, 0.849120082815735, 1.2932597556168703]


## 3. Verify Data Shapes

In [4]:
# Verify data shapes
print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")
print(f"Test data shape: {X_test.shape}")

Training data shape: (22967, 48, 48, 1)
Validation data shape: (5742, 48, 48, 1)
Test data shape: (7178, 48, 48, 1)


## 4. Initialize Model

In [5]:
# Create model
baseline_model = get_model('baseline', num_classes=7, device=device)

# Print architecture
print("Baseline CNN Architecture:")
print(baseline_model)

# Count parameters
total_params = sum(p.numel() for p in baseline_model.parameters())
trainable_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)

print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Baseline CNN Architecture:
BaselineCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4608, out_features=256, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=7, bias=True)
)

Total parameters: 1,274,823
Traina

## 5. Train Improved Model with Augmentation and Optimized Hyperparameters

In [6]:
# Create dataloaders with augmentation
train_loader, val_loader = create_dataloaders(
    X_train, y_train, 
    X_val, y_val, 
    batch_size=32, 
    device=device, 
    augment=True
)

print(f"✓ DataLoaders created")
print(f"  Training samples: {len(X_train)}")
print(f"  Validation samples: {len(X_val)}")
print(f"  Batch size: 32")
print(f"  Augmentation: Enabled (rotation, flip, translation, blur)")

✓ DataLoaders created
  Training samples: 22967
  Validation samples: 5742
  Batch size: 32
  Augmentation: Enabled (rotation, flip, translation, blur)


In [12]:
# Train improved baseline model with augmentation and optimized hyperparameters
history = train_model(
    baseline_model,
    train_loader,
    val_loader,
    epochs=150,
    learning_rate=0.0001,
    device=device,
    model_name='pytorch_baseline_cnn',
    class_weights=class_weights
)

print("\n✓ Training complete!")


Training pytorch_baseline_cnn
Device: cuda
Epoch     Train Loss     Train Acc      Val Loss       Val Acc        Learning Rate       
------------------------------------------------------------------------------------------------------------------------



5         1.7813         0.3224         1.6319         0.4181         1.0e-04             


10        1.6832         0.3732         1.5454         0.4302         1.0e-04             


15        1.6247         0.3967         1.4500         0.4768         1.0e-04             


20        1.5880         0.4119         1.4408         0.4838         1.0e-04             


...

EarlyStopping counter: 5/15


65        1.4078         0.4840         1.3202         0.5535         2.5e-05             


EarlyStopping counter: 10/15


70        1.3830         0.4848         1.2899         0.5507         2.5e-05             


EarlyStopping counter: 15/15
Early stopping at epoch 72 (best: 57)

Training stopped at epoch 73

Best model saved to saved_models/pytorch_baseline_cnn_best.pt
Final model saved to saved_models/pytorch_baseline_cnn_final.pt

✓ Training complete!


## 6. Plot Training History

In [13]:
# Plot and save training history
os.makedirs('../results/model', exist_ok=True)
plot_training_history(
    history,
    model_name='PyTorch Baseline CNN (Improved)',
    save_path='../results/model/pytorch_baseline_training_history.png'
)

Training history plot saved to ../results/model/pytorch_baseline_training_history.png


## 7. Evaluate on Test Set

In [17]:
# Load best model and evaluate on test set
baseline_model.load_state_dict(torch.load('saved_models/pytorch_baseline_cnn_best.pt', weights_only=True, map_location=device))

# Comprehensive evaluation
results = create_evaluation_report(
    baseline_model,
    X_test,
    y_test,
    device=device,
    model_name='pytorch_baseline'
)


Evaluating pytorch_baseline

Test Accuracy: 0.5371

Classification Report:
              precision    recall  f1-score   support

       Angry       0.42      0.37      0.40       958
     Disgust       0.22      0.59      0.33       111
        Fear       0.33      0.18      0.23      1024
       Happy       0.81      0.81      0.81      1774
     Neutral       0.49      0.56      0.52      1233
         Sad       0.39      0.36      0.38      1247
    Surprise       0.59      0.82      0.69       831

    accuracy                           0.54      7178
   macro avg       0.47      0.53      0.48      7178
weighted avg       0.53      0.54      0.52      7178

Confusion matrix saved to results/model/pytorch_pytorch_baseline_confusion_matrix.png
Per-class metrics plot saved to results/model/pytorch_pytorch_baseline_per_class_metrics.png
Prediction distribution plot saved to results/model/pytorch_pytorch_baseline_confidence.png

Evaluation complete for pytorch_baseline


## 8. Summary and Statistics

In [18]:
# Training and Evaluation Summary
print("="*70)
print("BASELINE CNN - FINAL RESULTS")
print("="*70)

print(f"\nModel Configuration:")
print(f"  Architecture: Baseline CNN with 3 conv blocks")
print(f"  Total Parameters: {total_params:,}")
print(f"  Device: {device}")

print(f"\nTraining Configuration:")
print(f"  Learning Rate: 0.0001")
print(f"  Batch Size: 32")
print(f"  Data Augmentation: Enabled (flip + brightness)")
if 'history' in locals():
    print(f"  Epochs Trained: {len(history['train_loss'])}")
    
    print(f"\nTraining Metrics:")
    print(f"  Best Val Accuracy: {max(history['val_accuracy']):.4f}")
    print(f"  Best Val Loss: {min(history['val_loss']):.4f}")
    print(f"  Final Train Accuracy: {history['train_accuracy'][-1]:.4f}")
else:
    print("  (Training history not yet available - run training cells first)")

print(f"\n{'='*70}")
print(f"TEST SET EVALUATION")
print(f"{'='*70}")
if 'results' in locals():
    print(f"  Test Accuracy: {results['accuracy']:.4f} ({results['accuracy']*100:.2f}%)")
    
    print(f"\nBest Performing Classes:")
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, results['y_pred'], average=None)
    top_classes = np.argsort(f1)[-3:][::-1]
    emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
    for idx in top_classes:
        print(f"  {emotion_labels[idx]:10s} - Precision: {precision[idx]:.3f}, Recall: {recall[idx]:.3f}, F1: {f1[idx]:.3f}")
else:
    print("  (Evaluation results not yet available - run evaluation cells first)")

print(f"\nModel Checkpoints:")
print(f"  Best Model: saved_models/pytorch_baseline_cnn_best.pt")
print(f"  Final Model: saved_models/pytorch_baseline_cnn_final.pt")

print(f"\nOutput Files:")
print(f"  Training History Plot: results/model/pytorch_baseline_training_history.png")
print(f"  Confusion Matrix: results/pytorch_pytorch_baseline_confusion_matrix.png")
print(f"  Per-Class Metrics: results/pytorch_pytorch_baseline_per_class_metrics.png")
print(f"  Confidence Distribution: results/pytorch_pytorch_baseline_confidence.png")
print(f"\n{'='*70}")

BASELINE CNN - FINAL RESULTS

Model Configuration:
  Architecture: Baseline CNN with 3 conv blocks
  Total Parameters: 1,274,823
  Device: cuda

Training Configuration:
  Learning Rate: 0.0001
  Batch Size: 32
  Data Augmentation: Enabled (flip + brightness)
  Epochs Trained: 73

Training Metrics:
  Best Val Accuracy: 0.5556
  Best Val Loss: 1.2640
  Final Train Accuracy: 0.4877

TEST SET EVALUATION
  Test Accuracy: 0.5371 (53.71%)

Best Performing Classes:
  Happy      - Precision: 0.809, Recall: 0.806, F1: 0.808
  Surprise   - Precision: 0.592, Recall: 0.816, F1: 0.686
  Neutral    - Precision: 0.486, Recall: 0.557, F1: 0.519

Model Checkpoints:
  Best Model: saved_models/pytorch_baseline_cnn_best.pt
  Final Model: saved_models/pytorch_baseline_cnn_final.pt

Output Files:
  Training History Plot: results/model/pytorch_baseline_training_history.png
  Confusion Matrix: results/pytorch_pytorch_baseline_confusion_matrix.png
  Per-Class Metrics: results/pytorch_pytorch_baseline_per_class_